# 02 · Очистка данных

Ноутбук очищает данные перед объединением и анализом.

- **Часть 1:** погода и модельный PM2.5 (CAMS) из Open-Meteo.
- **Часть 2:** PM2.5 станции (добавляется в задаче 2.2).

Порядок шагов — по Лекции 4: инспекция → имена → типы → дубликаты → пропуски → невозможные значения → выбросы → итоговые проверки.

Сырые файлы из `data/raw/` только читаем. Результат записываем в `data/processed/`. Запускать сверху вниз.

# Часть 1. Погода и CAMS

## Шаг 1. Импорты и пути

Подключаем pandas и наши модули `src/clean.py` и `src/collect.py`, задаём пути к файлам и списки колонок.

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src import clean, collect

WEATHER_RAW_PATH = "../data/raw/weather.csv"
CAMS_RAW_PATH = "../data/raw/cams_aq.csv"
WEATHER_CLEAN_PATH = "../data/processed/weather_clean.csv"
CAMS_CLEAN_PATH = "../data/processed/cams_clean.csv"

WEATHER_COLUMNS = list(collect.WEATHER_COLUMNS.values())
CAMS_COLUMNS = list(collect.CAMS_COLUMNS.values())

## Шаг 2. Инспекция

По Лекции 4 сначала смотрим на данные, потом исправляем. Подробный осмотр сделан в `01_collect` (шаги 12–15), здесь — короткая сводка: размер, типы, пропуски.

In [2]:
weather_raw = pd.read_csv(WEATHER_RAW_PATH)
cams_raw = pd.read_csv(CAMS_RAW_PATH)

for name, table in [("weather", weather_raw), ("cams", cams_raw)]:
    print(name, table.shape)
    print(table.dtypes)
    print(table.isna().sum())
    print()

weather_raw.describe()

weather (49104, 7)
time_utc          str
temp_c        float64
humidity        int64
precip        float64
pressure      float64
wind_speed    float64
wind_dir        int64
dtype: object
time_utc      0
temp_c        0
humidity      0
precip        0
pressure      0
wind_speed    0
wind_dir      0
dtype: int64

cams (28776, 2)
time_utc         str
cams_pm25    float64
dtype: object
time_utc     0
cams_pm25    0
dtype: int64



,temp_c,humidity,precip,pressure,wind_speed,wind_dir
count,49104.000000,49104.000000,49104.000000,49104.000000,49104.000000,49104.00000
mean,11.873153,55.073884,0.059883,1018.549723,1.563412,181.14897
std,11.913288,21.377138,0.323194,9.978545,0.910536,106.37092
min,-28.400000,7.000000,0.000000,994.600000,0.000000,1.00000
25%,2.400000,38.000000,0.000000,1010.800000,0.920000,101.00000
50%,13.000000,53.000000,0.000000,1017.500000,1.430000,169.00000
75%,21.300000,72.000000,0.000000,1025.400000,2.020000,270.00000
max,39.100000,100.000000,10.600000,1058.900000,10.740000,360.00000


## Шаг 3. Имена колонок

Имена проекта в snake_case заданы при выгрузке, значения не менялись. Модельный PM2.5 называется `cams_pm25`. Проверяем, что имена и порядок ожидаемые.

In [3]:
clean.check_columns(weather_raw, WEATHER_COLUMNS)
clean.check_columns(cams_raw, CAMS_COLUMNS)
print(list(weather_raw.columns))
print(list(cams_raw.columns))

['time_utc', 'temp_c', 'humidity', 'precip', 'pressure', 'wind_speed', 'wind_dir']
['time_utc', 'cams_pm25']


## Шаг 4. Типы: время в UTC

Сырые метки времени — строки без `Z`, поэтому `utc=True` и `format=` задаём явно. `errors="coerce"` превращает битые метки в NaT, а функция сразу считает их и падает с понятным сообщением. Числовые колонки уже числа.

In [4]:
weather = weather_raw.copy()
weather["time_utc"] = clean.parse_utc_times(weather_raw["time_utc"], collect.METEO_TIME_FORMAT)
cams = cams_raw.copy()
cams["time_utc"] = clean.parse_utc_times(cams_raw["time_utc"], collect.METEO_TIME_FORMAT)

for name, table in [("weather", weather), ("cams", cams)]:
    print(name, table["time_utc"].dtype, "NaT:", table["time_utc"].isna().sum())
    print("первая:", table["time_utc"].iloc[0], "последняя:", table["time_utc"].iloc[-1])

weather datetime64[us, UTC] NaT: 0
первая: 2020-04-09 00:00:00+00:00 последняя: 2025-11-14 23:00:00+00:00
cams datetime64[us, UTC] NaT: 0
первая: 2022-08-04 00:00:00+00:00 последняя: 2025-11-14 23:00:00+00:00


## Шаг 5. Местное время Алматы

Анализ ведём по `Asia/Almaty`. 1 марта 2024 пояс сменился с UTC+6 на UTC+5, поэтому смотрим стык: какие местные метки получаются из UTC-часов вокруг перехода.

In [5]:
junction = weather[
    (weather["time_utc"] >= pd.Timestamp("2024-02-29 16:00", tz="UTC"))
    & (weather["time_utc"] <= pd.Timestamp("2024-02-29 19:00", tz="UTC"))
]
pd.DataFrame({
    "time_utc": junction["time_utc"],
    "time_almaty": clean.to_almaty_time(junction["time_utc"]),
})

,time_utc,time_almaty
34120,2024-02-29 16:00:00+00:00,2024-02-29 22:00:00+06:00
34121,2024-02-29 17:00:00+00:00,2024-02-29 23:00:00+06:00
34122,2024-02-29 18:00:00+00:00,2024-02-29 23:00:00+05:00
34123,2024-02-29 19:00:00+00:00,2024-03-01 00:00:00+05:00


In [6]:
for name, table in [("weather", weather), ("cams", cams)]:
    local_naive = clean.to_almaty_time(table["time_utc"]).dt.tz_localize(None)
    print(name, "повторов местного времени без смещения:", local_naive.duplicated().sum())

weather повторов местного времени без смещения: 1
cams повторов местного времени без смещения: 1


Местная метка 23:00 29.02.2024 встречается дважды: один раз со смещением +06:00, второй раз +05:00. Поэтому ключом для объединения оставляем `time_utc` — он не прыгает. В файл местное время не пишем, а `clean.to_almaty_time` применяем при построении признаков (задача 3.1).

## Шаг 6. Дубликаты

Опасны дубликаты ключа: повторяющийся час сломает объединение. Считаем повторы `time_utc`.

In [7]:
print("weather дублей time_utc:", weather["time_utc"].duplicated().sum())
print("cams дублей time_utc:", cams["time_utc"].duplicated().sum())

weather дублей time_utc: 0
cams дублей time_utc: 0
